<div style="background:#0B3B36;background-image:linear-gradient(135deg,#0B3B36 0%,#12615A 60%,#1B7F72 100%);
border-radius:14px;padding:34px 38px;font-family:Arial,Helvetica,sans-serif;">
<div style="color:#4BD4B1;font-size:13px;font-weight:700;letter-spacing:3px;">ANALISA DATA LANJUTAN</div>
<div style="color:#FFFFFF;font-size:34px;font-weight:800;margin:10px 0 6px 0;line-height:1.2;">
Segmentasi Pelanggan &amp; RFM Analysis</div>
<div style="color:#BFE6DC;font-size:16px;line-height:1.6;">
Studi kasus: <b>Seduh Coffee</b> &mdash; brand kopi spesialti D2C Indonesia.<br>
Dari 24 ribu transaksi menjadi 7 segmen pelanggan dan rekomendasi bisnis yang actionable.</div>
<div style="color:#8FD8C6;font-size:13px;margin-top:18px;">Durasi 2 jam &nbsp;&middot;&nbsp; 12 latihan &nbsp;&middot;&nbsp; Google Colab + pandas</div>
</div>

<div style="font-family:Arial,Helvetica,sans-serif;color:#123331;">
<h3 style="margin-bottom:8px;">Cara memakai notebook ini</h3>
<table style="border-collapse:collapse;font-size:14px;">
<tr>
<td style="background:#E7F6F1;border-left:8px solid #0B3B36;padding:12px 16px;width:300px;">
<b style="color:#0B3B36;">&#9654;&nbsp; Kotak hijau</b><br>Dibahas bersama di kelas &mdash; ikuti alurnya, jalankan selnya.</td>
<td style="background:#FFF3E0;border-left:8px solid #C77700;padding:12px 16px;width:300px;">
<b style="color:#C77700;">&#9998;&nbsp; Kotak oranye</b><br>Kerjakan dulu sendiri (5&ndash;7 menit), baru kita bahas jawabannya.</td>
</tr>
</table>
<p style="font-size:14px;line-height:1.7;margin-top:14px;">
<b>Sebelum mulai:</b> jalankan <b>Sel Persiapan A</b> dan <b>B</b> di bawah sampai selesai.
Semua latihan bergantung pada dua sel itu. Kalau kernel Colab restart, jalankan ulang keduanya.<br>
<b>Tips:</b> <code>Ctrl+Enter</code> menjalankan sel, <code>Shift+Enter</code> menjalankan lalu pindah ke sel berikutnya.
</p></div>

In [ ]:
# ================== SEL PERSIAPAN A - Muat library & file dataset ==================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.options.display.float_format = "{:,.2f}".format

NAMA_FILE = "Seduh Coffee - Final Project (BNSP) - Sept 2026 v02.xlsx"

# Di Google Colab: kalau file belum ada, kotak upload akan muncul.
if not os.path.exists(NAMA_FILE):
    try:
        from google.colab import files
        print("Silakan pilih file:", NAMA_FILE)
        diupload = files.upload()
        NAMA_FILE = list(diupload.keys())[0]
    except ImportError:
        raise FileNotFoundError(
            f"'{NAMA_FILE}' tidak ditemukan. Letakkan file di folder yang sama dengan notebook ini."
        )

print("File siap dipakai:", NAMA_FILE)

# Palet warna kelas (mengikuti tema materi)
WARNA = {
    "tosca":  "#4BD4B1",
    "gelap":  "#0B3B36",
    "sedang": "#12615A",
    "muda":   "#BFE6DC",
    "oranye": "#E8833A",
    "merah":  "#C4553B",
    "abu":    "#7A8C89",
}
URUTAN_SEGMEN = [
    "Champions", "Loyal Customers", "Potential Loyalist", "Need Attention",
    "At Risk", "Can't Lose Them", "Hibernating",
]
WARNA_SEGMEN = {
    "Champions":          "#0B3B36",
    "Loyal Customers":    "#12615A",
    "Potential Loyalist": "#4BD4B1",
    "Need Attention":     "#BFE6DC",
    "At Risk":            "#E8833A",
    "Can't Lose Them":    "#C4553B",
    "Hibernating":        "#7A8C89",
}

def rp(x):
    """Format angka menjadi Rupiah singkat, mis. 3_008_035_000 -> 'Rp 3,01 M'."""
    x = float(x)
    if abs(x) >= 1_000_000_000:
        return f"Rp {x/1_000_000_000:,.2f} M".replace(",", "#").replace(".", ",").replace("#", ".")
    if abs(x) >= 1_000_000:
        return f"Rp {x/1_000_000:,.1f} jt".replace(",", "#").replace(".", ",").replace("#", ".")
    return f"Rp {x:,.0f}".replace(",", ".")

print("Sel Persiapan A selesai.")

In [ ]:
# ================== SEL PERSIAPAN B - Baca & rapikan data (sudah disiapkan) ==================
# Tidak ada yang perlu diubah di sel ini. Jalankan saja.
# Isinya adalah langkah pembersihan standar yang sudah kita pelajari di sesi sebelumnya,
# supaya waktu kelas hari ini bisa fokus penuh ke RFM & segmentasi.

orders_raw = pd.read_excel(NAMA_FILE, sheet_name="Orders")
customers  = pd.read_excel(NAMA_FILE, sheet_name="Customers")
products   = pd.read_excel(NAMA_FILE, sheet_name="Products")

# Nama channel di data mentah tidak konsisten: kapitalisasi, spasi depan/belakang, dan salah ketik.
RAPIKAN_CHANNEL = {
    "Tokopedia": "Tokopedia", "Tokopdia": "Tokopedia",
    "Shopee":    "Shopee",    "Shoppee":  "Shopee",
    "Webstore":  "Webstore",
    "Tiktok Shop": "TikTok Shop",
}

orders = (
    orders_raw
      .drop_duplicates()                                   # buang baris kembar persis
      .drop_duplicates(subset="order_id", keep="first")    # 1 order_id = 1 baris
      .assign(
          # sebagian order_date tersimpan sebagai TEKS "dd/mm/yyyy" -> dayfirst=True
          order_date=lambda d: pd.to_datetime(d["order_date"], dayfirst=True, errors="coerce"),
          quantity=lambda d: pd.to_numeric(d["quantity"], errors="coerce"),
          channel=lambda d: d["channel"].str.strip().str.title().map(RAPIKAN_CHANNEL),
          city=lambda d: d["city"].str.strip().str.title(),
      )
      .query("quantity > 0")                               # buang qty negatif / nol
      .query("order_status == 'Completed'")                # RFM hanya memakai transaksi sah
      .copy()
)

# Kolom turunan: nilai transaksi setelah diskon
orders["revenue"] = orders["quantity"] * orders["unit_price"] * (1 - orders["discount_pct"])

# Rapikan tabel Customers (untuk profiling & ekspor daftar target di akhir sesi)
RAPIKAN_GENDER = {
    "Female": "Female", "F": "Female", "female": "Female", "FEMALE": "Female", "Perempuan": "Female",
    "Male": "Male", "M": "Male", "male": "Male", "MALE": "Male", "Laki-laki": "Male",
}
customers = customers.assign(
    gender=lambda d: d["gender"].map(RAPIKAN_GENDER),
    customer_name=lambda d: d["customer_name"].str.strip()
                             .str.replace(r"\s+", " ", regex=True).str.title(),
    city=lambda d: d["city"].fillna("Tidak diketahui").str.strip().str.title(),
)

print(f"Baris order mentah    : {len(orders_raw):,}")
print(f"Baris order siap pakai: {len(orders):,}")
print(f"Rentang tanggal       : {orders['order_date'].min():%d %b %Y} s/d {orders['order_date'].max():%d %b %Y}")
print("Sel Persiapan B selesai.")

<div style="background:#E7F6F1;border:1px solid #0B3B36;border-left:8px solid #0B3B36;border-radius:10px;padding:16px 20px;margin:4px 0;font-family:Arial,Helvetica,sans-serif;color:#123331;"><div style="font-size:12px;font-weight:700;letter-spacing:1.5px;color:#0B3B36;margin-bottom:6px;">&#9654;&nbsp;&nbsp;LATIHAN 1</div><div style="font-size:19px;font-weight:700;color:#123331;margin-bottom:10px;">Kenali dulu datanya sebelum menghitung apa pun</div><div style="font-size:14px;line-height:1.65;">Sebelum menghitung RFM, kita harus tahu <b>satu baris data itu artinya apa</b>.
Di sini kita cek tiga hal: ukuran tabel, contoh isinya, dan rentang tanggalnya.<br><br>
<b>Yang kita lakukan:</b>
<ol>
<li>Tampilkan jumlah baris &amp; kolom dari <code>orders</code>, <code>customers</code>, <code>products</code>.</li>
<li>Lihat 5 baris pertama <code>orders</code>.</li>
<li>Hitung berapa pelanggan unik yang benar-benar pernah bertransaksi.</li>
</ol>
<b>Pertanyaan pemantik:</b> kolom mana yang nanti dipakai untuk <b>R</b>, <b>F</b>, dan <b>M</b>?</div></div>

In [ ]:
# 1a. Ukuran tiap tabel
print("orders   :", orders.shape)
print("customers:", customers.shape)
print("products :", products.shape)

# 1b. Lihat isinya
display(orders.head())

# 1c. Berapa pelanggan yang pernah bertransaksi?
print("\nPelanggan unik di orders   :", orders["customer_id"].nunique())
print("Pelanggan terdaftar         :", customers["customer_id"].nunique())
print("Belum pernah belanja        :",
      customers["customer_id"].nunique() - orders["customer_id"].nunique())

<div style="background:#FFF3E0;border:1px solid #C77700;border-left:8px solid #C77700;border-radius:10px;padding:16px 20px;margin:4px 0;font-family:Arial,Helvetica,sans-serif;color:#123331;"><div style="font-size:12px;font-weight:700;letter-spacing:1.5px;color:#C77700;margin-bottom:6px;">&#9998;&nbsp;&nbsp;LATIHAN 2</div><div style="font-size:19px;font-weight:700;color:#123331;margin-bottom:10px;">Potret bisnis dalam 5 angka</div><div style="font-size:14px;line-height:1.65;">Sekarang giliran Anda. Dari tabel <code>orders</code>, hitung dan tampilkan:
<ol>
<li><b>Total revenue</b> seluruh periode.</li>
<li><b>Jumlah order</b> (baris unik <code>order_id</code>).</li>
<li><b>Jumlah pelanggan unik</b>.</li>
<li><b>AOV</b> (<i>Average Order Value</i>) = total revenue &divide; jumlah order.</li>
<li><b>Revenue per <code>channel</code></b>, diurutkan dari terbesar, lengkap dengan kolom kontribusi (%).</li>
</ol>
<b>Bonus:</b> channel mana yang order-nya banyak tetapi AOV-nya justru paling kecil?</div></div>

In [ ]:
# TODO 2a-2d: lengkapi lima angka ringkasan di bawah
total_revenue = ____
jumlah_order  = ____
jumlah_pelanggan = ____
aov = ____

print("Total revenue    :", rp(total_revenue))
print("Jumlah order     :", f"{jumlah_order:,}")
print("Jumlah pelanggan :", f"{jumlah_pelanggan:,}")
print("AOV              :", rp(aov))

# TODO 2e: revenue per channel + kontribusi persen + AOV per channel
per_channel = (
    orders.groupby("____")
          .agg(revenue=("____", "____"),
               order=("____", "____"))
)
per_channel["kontribusi_%"] = ____
per_channel["aov"] = ____
display(per_channel.sort_values("revenue", ascending=False).round(1))

<div style="font-family:Arial,Helvetica,sans-serif;font-size:13px;color:#7A8C89;"><i>Catatan Anda untuk Latihan 2:</i></div>

*(tulis kesimpulan singkat Anda di sini)*

<div style="background:#E7F6F1;border:1px solid #0B3B36;border-left:8px solid #0B3B36;border-radius:10px;padding:16px 20px;margin:4px 0;font-family:Arial,Helvetica,sans-serif;color:#123331;"><div style="font-size:12px;font-weight:700;letter-spacing:1.5px;color:#0B3B36;margin-bottom:6px;">&#9654;&nbsp;&nbsp;LATIHAN 3</div><div style="font-size:19px;font-weight:700;color:#123331;margin-bottom:10px;">Menetapkan snapshot date & menghitung Recency</div><div style="font-size:14px;line-height:1.65;"><b>Recency</b> = sudah berapa hari sejak pelanggan terakhir bertransaksi.
Supaya bisa dihitung, kita butuh satu titik acuan: <b>snapshot date</b>.<br><br>
Konvensi yang lazim: <code>snapshot = tanggal transaksi terakhir + 1 hari</code>.
Kenapa +1? Supaya pelanggan yang belanja tepat di hari terakhir mendapat Recency = 1, bukan 0
(nol menyulitkan saat dibagi atau di-log).<br><br>
<b>Ingat:</b> Recency itu <i>semakin kecil semakin baik</i>. Ini satu-satunya dari R/F/M yang arahnya terbalik.</div></div>

In [ ]:
# 3a. Tentukan snapshot date
snapshot_date = orders["order_date"].max() + pd.Timedelta(days=1)
print("Transaksi terakhir :", orders["order_date"].max().strftime("%d %b %Y"))
print("Snapshot date      :", snapshot_date.strftime("%d %b %Y"))

# 3b. Tanggal transaksi TERAKHIR tiap pelanggan
terakhir = orders.groupby("customer_id")["order_date"].max()

# 3c. Recency = selisih hari terhadap snapshot date
recency = (snapshot_date - terakhir).dt.days
recency.name = "Recency"

display(recency.head())
print("\nRecency  min:", recency.min(), "hari | median:", int(recency.median()),
      "hari | max:", recency.max(), "hari")

<div style="background:#FFF3E0;border:1px solid #C77700;border-left:8px solid #C77700;border-radius:10px;padding:16px 20px;margin:4px 0;font-family:Arial,Helvetica,sans-serif;color:#123331;"><div style="font-size:12px;font-weight:700;letter-spacing:1.5px;color:#C77700;margin-bottom:6px;">&#9998;&nbsp;&nbsp;LATIHAN 4</div><div style="font-size:19px;font-weight:700;color:#123331;margin-bottom:10px;">Melengkapi tabel RFM: Frequency & Monetary</div><div style="font-size:14px;line-height:1.65;">Recency sudah ada. Sekarang lengkapi dua sisanya dan gabungkan menjadi satu tabel bernama <code>rfm</code>
dengan kolom: <code>customer_id, Recency, Frequency, Monetary</code>.<br><br>
<b>Definisi yang dipakai:</b>
<ul>
<li><b>Frequency</b> = banyaknya <i>order unik</i> pelanggan tersebut &rarr; gunakan <code>nunique</code> pada <code>order_id</code>, bukan <code>count</code>.</li>
<li><b>Monetary</b> = total <code>revenue</code> pelanggan tersebut selama seluruh periode.</li>
</ul>
<b>Tantangan:</b> kerjakan ketiganya sekaligus dalam <b>satu</b> <code>groupby().agg()</code> — jangan tiga groupby terpisah.<br>
Lalu tampilkan <code>rfm.describe()</code> dan jawab: berapa median Frequency? Apakah distribusi Monetary simetris atau menceng?</div></div>

In [ ]:
# TODO 4: bangun tabel rfm lengkap dalam SATU groupby().agg()
rfm = (
    orders.groupby("customer_id")
          .agg(
              Recency   = ("order_date", lambda s: ____),
              Frequency = ("____", "____"),
              Monetary  = ("____", "____"),
          )
          .reset_index()
)

print("Jumlah baris rfm:", len(rfm), "(harus sama dengan jumlah pelanggan unik)")
display(rfm.head())
display(rfm[["Recency", "Frequency", "Monetary"]].describe().round(1))

<div style="font-family:Arial,Helvetica,sans-serif;font-size:13px;color:#7A8C89;"><i>Catatan Anda untuk Latihan 4:</i></div>

*(tulis kesimpulan singkat Anda di sini)*

<div style="background:#E7F6F1;border:1px solid #0B3B36;border-left:8px solid #0B3B36;border-radius:10px;padding:16px 20px;margin:4px 0;font-family:Arial,Helvetica,sans-serif;color:#123331;"><div style="font-size:12px;font-weight:700;letter-spacing:1.5px;color:#0B3B36;margin-bottom:6px;">&#9654;&nbsp;&nbsp;LATIHAN 5</div><div style="font-size:19px;font-weight:700;color:#123331;margin-bottom:10px;">Dari angka mentah ke skor 1-5: R_Score</div><div style="font-size:14px;line-height:1.65;">Recency 12 hari dan Monetary Rp 3 juta tidak bisa dibandingkan langsung — satuannya beda.
Solusinya: ubah keduanya menjadi <b>skor 1&ndash;5</b> berdasarkan <b>kuintil</b> (5 kelompok sama besar) dengan <code>pd.qcut</code>.<br><br>
<b>Kuncinya ada di arah label:</b>
<ul>
<li>Recency kecil = <b>bagus</b> &rarr; label dibalik <code>[5, 4, 3, 2, 1]</code></li>
<li>Frequency &amp; Monetary besar = bagus &rarr; label normal <code>[1, 2, 3, 4, 5]</code></li>
</ul>
Setelah ini kita cek batas tiap kuintil, supaya skor punya arti bisnis yang bisa dijelaskan ke stakeholder.</div></div>

In [ ]:
# 5a. Skor Recency - perhatikan urutan label yang DIBALIK
rfm["R_Score"] = pd.qcut(rfm["Recency"], q=5, labels=[5, 4, 3, 2, 1]).astype(int)

# 5b. Cek batas tiap kuintil supaya skornya bisa dijelaskan ke bisnis
batas_r = rfm.groupby("R_Score")["Recency"].agg(["min", "max", "count"])
batas_r.index.name = "R_Score"
display(batas_r)

print("\nTerjemahan bisnisnya:")
for skor in [5, 4, 3, 2, 1]:
    b = batas_r.loc[skor]
    print(f"  R_Score {skor}: belanja terakhir {int(b['min'])}-{int(b['max'])} hari lalu "
          f"({int(b['count'])} pelanggan)")

<div style="background:#FFF3E0;border:1px solid #C77700;border-left:8px solid #C77700;border-radius:10px;padding:16px 20px;margin:4px 0;font-family:Arial,Helvetica,sans-serif;color:#123331;"><div style="font-size:12px;font-weight:700;letter-spacing:1.5px;color:#C77700;margin-bottom:6px;">&#9998;&nbsp;&nbsp;LATIHAN 6</div><div style="font-size:19px;font-weight:700;color:#123331;margin-bottom:10px;">F_Score, M_Score, dan RFM Score gabungan</div><div style="font-size:14px;line-height:1.65;">Lengkapi sistem skornya. Buat empat kolom baru di <code>rfm</code>:
<ol>
<li><code>F_Score</code> dan <code>M_Score</code> &mdash; skala 1&ndash;5, arah label <b>normal</b> (makin besar makin baik).</li>
<li><code>RFM_Score</code> = R_Score + F_Score + M_Score (rentang 3&ndash;15). Ini dipakai untuk <i>ranking</i> pelanggan.</li>
<li><code>RFM_Cell</code> = ketiga skor <b>disambung sebagai teks</b>, mis. <code>"545"</code>. Ini dipakai untuk <i>pola perilaku</i>.</li>
</ol>
<div style="background:#FFFFFF;border:1px dashed #C77700;border-radius:8px;padding:12px 14px;margin-top:10px;">
<b>Jebakan yang harus Anda tangani:</b> <code>pd.qcut(rfm["Frequency"], 5, ...)</code> akan <b>error</b>
(<i>"Bin edges must be unique"</i>). Penyebabnya: <b>62% pelanggan Seduh hanya belanja satu kali</b>,
sehingga empat dari lima batas kuintil jatuh di angka yang sama.<br>
<b>Petunjuk:</b> jangan dipaksakan memakai kuintil. Tetapkan <code>F_Score</code> dari <b>jumlah order</b>:
1&nbsp;order&nbsp;&rarr;&nbsp;1, 2&nbsp;&rarr;&nbsp;2, 3&nbsp;&rarr;&nbsp;3, 4&ndash;5&nbsp;&rarr;&nbsp;4, &ge;6&nbsp;&rarr;&nbsp;5.
Monetary nilainya kontinu, jadi <code>qcut</code> tetap aman dipakai di sana.
</div>
<br><b>Terakhir:</b> tampilkan 5 pelanggan dengan RFM_Score tertinggi.</div></div>

In [ ]:
# TODO 6a: F_Score - kuintil akan gagal di sini. Coba dulu baris berikut supaya
#          teman-teman melihat sendiri error "Bin edges must be unique":
# rfm["F_Score"] = pd.qcut(rfm["Frequency"], q=5, labels=[1, 2, 3, 4, 5])

def skor_frequency(jumlah_order):
    """Ambang bisnis: >=6 -> 5, 4-5 -> 4, 3 -> 3, 2 -> 2, 1 -> 1."""
    if jumlah_order >= 6:
        return 5
    # ... lanjutkan ...
    return ____

rfm["F_Score"] = rfm["Frequency"].apply(____)

# TODO 6b: M_Score - Monetary kontinu, qcut aman
rfm["M_Score"] = pd.qcut(____, q=5, labels=[____]).astype(int)

# TODO 6c: skor gabungan (angka) dan pola sel (teks)
rfm["RFM_Score"] = ____
rfm["RFM_Cell"]  = ____

display(rfm.head())

# TODO 6d: 5 pelanggan terbaik
display(rfm.sort_values("____", ascending=False).head(5))

# Cek sebaran F_Score
print("\nSebaran F_Score:")
print(rfm.groupby("F_Score")["Frequency"].agg(["min", "max", "count"]))

<div style="font-family:Arial,Helvetica,sans-serif;font-size:13px;color:#7A8C89;"><i>Catatan Anda untuk Latihan 6:</i></div>

*(tulis kesimpulan singkat Anda di sini)*

<div style="background:#E7F6F1;border:1px solid #0B3B36;border-left:8px solid #0B3B36;border-radius:10px;padding:16px 20px;margin:4px 0;font-family:Arial,Helvetica,sans-serif;color:#123331;"><div style="font-size:12px;font-weight:700;letter-spacing:1.5px;color:#0B3B36;margin-bottom:6px;">&#9654;&nbsp;&nbsp;LATIHAN 7</div><div style="font-size:19px;font-weight:700;color:#123331;margin-bottom:10px;">Dari skor ke nama segmen yang dimengerti bisnis</div><div style="font-size:14px;line-height:1.65;">Tim marketing tidak akan menjalankan kampanye untuk "pelanggan RFM_Cell 545".
Mereka butuh <b>nama</b>. Jadi kita terjemahkan kombinasi skor menjadi segmen bernama, memakai aturan berbasis
<b>R</b> (masih aktif atau tidak) dan <b>F</b> (seberapa lekat).<br><br>
Di sini kita bangun kerangkanya dengan <b>3 aturan</b> lebih dulu, lalu Anda yang melengkapinya menjadi 7.<br><br>
<b>Perhatikan:</b> urutan <code>if</code> menentukan hasil. Aturan yang paling spesifik harus diletakkan paling atas.</div></div>

In [ ]:
def segmentasi_demo(baris):
    """Versi awal - baru 3 aturan. Kita lengkapi di latihan berikutnya."""
    r, f, m = baris["R_Score"], baris["F_Score"], baris["M_Score"]

    if r >= 4 and f >= 4 and m >= 4:        # paling spesifik -> paling atas
        return "Champions"
    if r >= 3 and f >= 3:
        return "Loyal Customers"
    if r <= 2 and f <= 2:
        return "Hibernating"
    return "Belum terklasifikasi"

rfm["Segment_Demo"] = rfm.apply(segmentasi_demo, axis=1)
display(rfm["Segment_Demo"].value_counts().to_frame("jumlah_pelanggan"))

sisa = (rfm["Segment_Demo"] == "Belum terklasifikasi").sum()
print(f"\nMasih ada {sisa:,} pelanggan ({sisa/len(rfm):.1%}) yang belum punya nama segmen.")

<div style="background:#FFF3E0;border:1px solid #C77700;border-left:8px solid #C77700;border-radius:10px;padding:16px 20px;margin:4px 0;font-family:Arial,Helvetica,sans-serif;color:#123331;"><div style="font-size:12px;font-weight:700;letter-spacing:1.5px;color:#C77700;margin-bottom:6px;">&#9998;&nbsp;&nbsp;LATIHAN 8</div><div style="font-size:19px;font-weight:700;color:#123331;margin-bottom:10px;">Melengkapi peta segmen menjadi 7 kelompok</div><div style="font-size:14px;line-height:1.65;">Lengkapi fungsi tadi menjadi <b>7 segmen</b>, sesuai definisi berikut
(urutkan aturannya sendiri &mdash; ini bagian yang menentukan):
<table style="border-collapse:collapse;font-size:13.5px;margin:10px 0;background:#FFFFFF;">
<tr style="background:#0B3B36;color:#FFFFFF;"><th style="padding:7px 12px;text-align:left;">Segmen</th><th style="padding:7px 12px;text-align:left;">Aturan</th><th style="padding:7px 12px;text-align:left;">Maknanya</th></tr>
<tr><td style="padding:6px 12px;"><b>Champions</b></td><td style="padding:6px 12px;">R&ge;4 <i>dan</i> F&ge;4 <i>dan</i> M&ge;4</td><td style="padding:6px 12px;">Baru belanja, sering, nilainya besar</td></tr>
<tr style="background:#F5FAF8;"><td style="padding:6px 12px;"><b>Loyal Customers</b></td><td style="padding:6px 12px;">R&ge;3 <i>dan</i> F&ge;3</td><td style="padding:6px 12px;">Aktif dan konsisten</td></tr>
<tr><td style="padding:6px 12px;"><b>Potential Loyalist</b></td><td style="padding:6px 12px;">R&ge;4 <i>dan</i> F&le;2</td><td style="padding:6px 12px;">Baru belanja, belum jadi kebiasaan</td></tr>
<tr style="background:#F5FAF8;"><td style="padding:6px 12px;"><b>Can't Lose Them</b></td><td style="padding:6px 12px;">R&le;2 <i>dan</i> F&ge;4</td><td style="padding:6px 12px;">Dulu sangat aktif, sekarang menghilang</td></tr>
<tr><td style="padding:6px 12px;"><b>At Risk</b></td><td style="padding:6px 12px;">R&le;2 <i>dan</i> F=3</td><td style="padding:6px 12px;">Mulai menjauh</td></tr>
<tr style="background:#F5FAF8;"><td style="padding:6px 12px;"><b>Hibernating</b></td><td style="padding:6px 12px;">R&le;2 <i>dan</i> F&le;2</td><td style="padding:6px 12px;">Lama tidak muncul, dulu pun jarang</td></tr>
<tr><td style="padding:6px 12px;"><b>Need Attention</b></td><td style="padding:6px 12px;">selain di atas</td><td style="padding:6px 12px;">Kelompok tengah</td></tr>
</table>
Simpan hasilnya di kolom <code>Segment</code>, lalu:
<ol>
<li>Pastikan <b>tidak ada</b> pelanggan yang tersisa tanpa segmen (<code>assert</code>).</li>
<li>Tampilkan jumlah <b>dan</b> persentase pelanggan per segmen, diurutkan dari terbanyak.</li>
</ol></div></div>

In [ ]:
def segmentasi(baris):
    """Terjemahkan R/F/M Score menjadi nama segmen. Urutan aturan menentukan hasil!"""
    r, f, m = baris["R_Score"], baris["F_Score"], baris["M_Score"]

    # TODO 8a: susun 7 aturan di sini, dari yang paling spesifik ke paling umum
    if ____:
        return "Champions"
    # ... lanjutkan ...

    return "Need Attention"   # penampung terakhir

rfm["Segment"] = rfm.apply(segmentasi, axis=1)

# TODO 8b: validasi - tidak boleh ada yang kosong / tak dikenal
assert rfm["Segment"].notna().all(), "Masih ada pelanggan tanpa segmen!"
assert rfm["Segment"].nunique() == 7, "Jumlah segmen belum 7"

# TODO 8c: jumlah + persentase per segmen
ringkas_segmen = ____
display(ringkas_segmen)

<div style="font-family:Arial,Helvetica,sans-serif;font-size:13px;color:#7A8C89;"><i>Catatan Anda untuk Latihan 8:</i></div>

*(tulis kesimpulan singkat Anda di sini)*

<div style="background:#E7F6F1;border:1px solid #0B3B36;border-left:8px solid #0B3B36;border-radius:10px;padding:16px 20px;margin:4px 0;font-family:Arial,Helvetica,sans-serif;color:#123331;"><div style="font-size:12px;font-weight:700;letter-spacing:1.5px;color:#0B3B36;margin-bottom:6px;">&#9654;&nbsp;&nbsp;LATIHAN 9</div><div style="font-size:19px;font-weight:700;color:#123331;margin-bottom:10px;">Visualisasi 1 - Berapa besar tiap segmen?</div><div style="font-size:14px;line-height:1.65;">Angka di tabel sulit dirasakan; bar chart membuat perbandingan langsung terlihat.
Kita buat <b>horizontal bar chart</b> jumlah pelanggan per segmen, dengan warna yang <b>konsisten</b> per segmen
supaya bisa dipakai ulang di semua chart berikutnya.<br><br>
<b>Tiga kebiasaan baik yang dipraktikkan di sini:</b> urutkan datanya, beri label nilai langsung di ujung bar,
dan buang garis bingkai yang tidak perlu.</div></div>

In [ ]:
jumlah = rfm["Segment"].value_counts().reindex(URUTAN_SEGMEN)

fig, ax = plt.subplots(figsize=(10, 5.2))
bars = ax.barh(
    jumlah.index[::-1], jumlah.values[::-1],
    color=[WARNA_SEGMEN[s] for s in jumlah.index[::-1]],
)

# Label nilai langsung di ujung bar - lebih mudah dibaca daripada sumbu X
for bar, nilai in zip(bars, jumlah.values[::-1]):
    ax.text(bar.get_width() + 12, bar.get_y() + bar.get_height() / 2,
            f"{nilai:,}  ({nilai/len(rfm):.0%})",
            va="center", fontsize=10, color="#123331")

ax.set_title("Jumlah Pelanggan per Segmen RFM", fontsize=15, fontweight="bold",
             color="#0B3B36", pad=14)
ax.set_xlabel("Jumlah pelanggan")
ax.set_xlim(0, jumlah.max() * 1.22)
ax.xaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))
for sisi in ["top", "right", "left"]:
    ax.spines[sisi].set_visible(False)
ax.tick_params(axis="y", length=0)
plt.tight_layout()
plt.show()

<div style="background:#FFF3E0;border:1px solid #C77700;border-left:8px solid #C77700;border-radius:10px;padding:16px 20px;margin:4px 0;font-family:Arial,Helvetica,sans-serif;color:#123331;"><div style="font-size:12px;font-weight:700;letter-spacing:1.5px;color:#C77700;margin-bottom:6px;">&#9998;&nbsp;&nbsp;LATIHAN 10</div><div style="font-size:19px;font-weight:700;color:#123331;margin-bottom:10px;">Visualisasi 2 - Jumlah orang vs besar uang</div><div style="font-size:14px;line-height:1.65;">Sekarang buat <b>dua</b> visual yang lebih menantang, dalam satu figure berdampingan
(<code>plt.subplots(1, 2, ...)</code>):<br><br>
<b>(a) Kiri &mdash; Pareto segmen.</b> Grouped bar chart yang membandingkan <b>% pelanggan</b> vs
<b>% revenue</b> untuk tiap segmen, diurutkan berdasarkan % revenue.
Inilah chart yang paling sering dibawa ke rapat manajemen.<br><br>
<b>(b) Kanan &mdash; Peta R &times; F.</b> Heatmap jumlah pelanggan untuk tiap kombinasi
<code>R_Score</code> (sumbu Y, 5 di atas) &times; <code>F_Score</code> (sumbu X).
Gunakan <code>pd.crosstab</code> lalu <code>ax.imshow</code>, dan tuliskan angkanya di tiap kotak.<br><br>
<b>Pertanyaan yang harus bisa Anda jawab dari chart (a):</b> segmen mana yang porsi revenue-nya
jauh <i>lebih besar</i> daripada porsi jumlah orangnya? Dan mana yang sebaliknya?</div></div>

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# ---------- (a) Pareto: % pelanggan vs % revenue ----------
# TODO 10a: bangun tabel bantu berisi persen pelanggan & persen revenue per segmen
pareto = (
    rfm.groupby("Segment")
       .agg(pelanggan=("customer_id", "____"),
            revenue=("Monetary", "____"))
)
pareto["pct_pelanggan"] = ____
pareto["pct_revenue"]   = ____
pareto = pareto.sort_values("pct_revenue")

y = np.arange(len(pareto))
tinggi = 0.38
# TODO 10b: dua set bar berdampingan (gunakan y - tinggi/2 dan y + tinggi/2)
ax1.barh(____, ____, height=tinggi, label="% pelanggan", color=WARNA["muda"])
ax1.barh(____, ____, height=tinggi, label="% revenue",   color=WARNA["gelap"])
ax1.set_yticks(y); ax1.set_yticklabels(pareto.index)
ax1.set_title("Porsi Pelanggan vs Porsi Revenue", fontsize=14, fontweight="bold", color="#0B3B36")
ax1.set_xlabel("Persen (%)"); ax1.legend(frameon=False, loc="lower right")
for sisi in ["top", "right", "left"]:
    ax1.spines[sisi].set_visible(False)

# ---------- (b) Heatmap R x F ----------
# TODO 10c: tabel silang jumlah pelanggan per kombinasi R_Score x F_Score
matriks = pd.crosstab(____, ____)
matriks = matriks.sort_index(ascending=False)   # R_Score 5 di baris atas

gambar = ax2.imshow(matriks.values, cmap="BuGn", aspect="auto")
ax2.set_xticks(range(len(matriks.columns))); ax2.set_xticklabels(matriks.columns)
ax2.set_yticks(range(len(matriks.index)));   ax2.set_yticklabels(matriks.index)
ax2.set_xlabel("F_Score  (makin ke kanan makin sering belanja)")
ax2.set_ylabel("R_Score  (makin ke atas makin baru belanja)")
ax2.set_title("Peta Sebaran Pelanggan: Recency x Frequency",
              fontsize=14, fontweight="bold", color="#0B3B36")

# TODO 10d: tulis angka di setiap kotak heatmap
for i in range(matriks.shape[0]):
    for j in range(matriks.shape[1]):
        nilai = ____
        ax2.text(j, i, f"{nilai}", ha="center", va="center",
                 color="white" if nilai > matriks.values.max() * 0.55 else "#0B3B36",
                 fontsize=11, fontweight="bold")

plt.tight_layout()
plt.show()

<div style="font-family:Arial,Helvetica,sans-serif;font-size:13px;color:#7A8C89;"><i>Catatan Anda untuk Latihan 10:</i></div>

*(tulis kesimpulan singkat Anda di sini)*

<div style="background:#E7F6F1;border:1px solid #0B3B36;border-left:8px solid #0B3B36;border-radius:10px;padding:16px 20px;margin:4px 0;font-family:Arial,Helvetica,sans-serif;color:#123331;"><div style="font-size:12px;font-weight:700;letter-spacing:1.5px;color:#0B3B36;margin-bottom:6px;">&#9654;&nbsp;&nbsp;LATIHAN 11</div><div style="font-size:19px;font-weight:700;color:#123331;margin-bottom:10px;">Tabel profil segmen - satu tabel untuk dibawa ke rapat</div><div style="font-size:14px;line-height:1.65;">Chart bagus untuk menyadarkan; tabel dibutuhkan untuk mengambil keputusan.
Kita rangkum tiap segmen menjadi satu baris berisi: jumlah pelanggan, rata-rata R/F/M,
total revenue, dan kontribusinya.<br><br>
Setelah tabel jadi, kita tambahkan satu kolom turunan yang sering dilupakan:
<b>revenue per pelanggan</b> &mdash; ukuran seberapa "mahal" satu kepala di segmen tersebut.</div></div>

In [ ]:
profil = (
    rfm.groupby("Segment")
       .agg(
           jumlah_pelanggan = ("customer_id", "count"),
           avg_recency      = ("Recency", "mean"),
           avg_frequency    = ("Frequency", "mean"),
           avg_monetary     = ("Monetary", "mean"),
           total_revenue    = ("Monetary", "sum"),
       )
       .sort_values("total_revenue", ascending=False)
)

profil["pct_pelanggan"]   = profil["jumlah_pelanggan"] / profil["jumlah_pelanggan"].sum() * 100
profil["pct_revenue"]     = profil["total_revenue"]    / profil["total_revenue"].sum()    * 100
profil["revenue_per_org"] = profil["total_revenue"]    / profil["jumlah_pelanggan"]

tampil = profil.copy()
tampil["avg_recency"]     = tampil["avg_recency"].round(0).astype(int).astype(str) + " hari"
tampil["avg_frequency"]   = tampil["avg_frequency"].round(1)
tampil["avg_monetary"]    = tampil["avg_monetary"].apply(rp)
tampil["total_revenue"]   = tampil["total_revenue"].apply(rp)
tampil["revenue_per_org"] = tampil["revenue_per_org"].apply(rp)
tampil["pct_pelanggan"]   = tampil["pct_pelanggan"].round(1)
tampil["pct_revenue"]     = tampil["pct_revenue"].round(1)

display(tampil)

<div style="background:#FFF3E0;border:1px solid #C77700;border-left:8px solid #C77700;border-radius:10px;padding:16px 20px;margin:4px 0;font-family:Arial,Helvetica,sans-serif;color:#123331;"><div style="font-size:12px;font-weight:700;letter-spacing:1.5px;color:#C77700;margin-bottom:6px;">&#9998;&nbsp;&nbsp;LATIHAN 12</div><div style="font-size:19px;font-weight:700;color:#123331;margin-bottom:10px;">Dari segmen menjadi keputusan bisnis</div><div style="font-size:14px;line-height:1.65;">Latihan penutup &mdash; bagian yang membedakan analis data dari operator tool.
Kerjakan empat hal:<br>
<ol>
<li><b>Hitung <i>revenue at risk</i></b>: total Monetary segmen <code>At Risk</code> + <code>Can't Lose Them</code>,
beserta persentasenya terhadap total revenue.</li>
<li><b>Hitung potensi <i>upside</i></b>: seandainya rata-rata Frequency segmen <code>Potential Loyalist</code>
naik dari nilai sekarang menjadi <b>2 order</b> (sekadar membuat mereka belanja kedua kalinya),
berapa tambahan revenue-nya? (asumsi: nilai per order segmen itu tetap).</li>
<li><b>Ekspor daftar target kampanye</b>: file CSV berisi pelanggan <code>Can't Lose Them</code>
&mdash; kolom <code>customer_id, customer_name, city, Recency, Frequency, Monetary</code> &mdash;
diurutkan dari Monetary terbesar. (Gabungkan dengan tabel <code>customers</code>.)</li>
<li><b>Tulis 3 rekomendasi</b> di sel Markdown terakhir. Tiap rekomendasi wajib memuat:
<i>segmen sasaran &rarr; aksi konkret &rarr; angka pendukung &rarr; metrik keberhasilan</i>.</li>
</ol>
<b>Aturan main:</b> rekomendasi tanpa angka dari analisis Anda sendiri tidak dihitung.</div></div>

In [ ]:
# TODO 12a: revenue at risk
segmen_berisiko = ["At Risk", "Can't Lose Them"]
revenue_at_risk = ____
pct_at_risk     = ____
print("Revenue at risk :", rp(revenue_at_risk), f"({pct_at_risk:.1f}% dari total)")

# TODO 12b: potensi upside dari Potential Loyalist (target: order KEDUA)
pl = rfm[rfm["Segment"] == "Potential Loyalist"]
nilai_per_order   = ____          # rata-rata Monetary dibagi rata-rata Frequency
frekuensi_target  = 2
tambahan_revenue  = ____
print("Potensi tambahan revenue:", rp(tambahan_revenue))

# TODO 12c: ekspor daftar target kampanye
target = (
    rfm[rfm["Segment"] == "____"]
      .merge(customers[["customer_id", "customer_name", "city"]], on="____", how="left")
      [["customer_id", "customer_name", "city", "Recency", "Frequency", "Monetary"]]
      .sort_values("____", ascending=False)
)
target.to_csv("target_winback_cant_lose_them.csv", index=False)
print("File tersimpan. Jumlah target:", len(target))
display(target.head(10))

<div style="font-family:Arial,Helvetica,sans-serif;font-size:13px;color:#7A8C89;"><i>Catatan Anda untuk Latihan 12:</i></div>

*(tulis kesimpulan singkat Anda di sini)*

<div style="background:#0B3B36;border-radius:14px;padding:30px 34px;font-family:Arial,Helvetica,sans-serif;">
<div style="color:#4BD4B1;font-size:12px;font-weight:700;letter-spacing:3px;">PENUTUP</div>
<div style="color:#FFFFFF;font-size:26px;font-weight:800;margin:8px 0 14px 0;">Yang kita bawa pulang</div>
<div style="color:#BFE6DC;font-size:15px;line-height:1.85;">
<b style="color:#FFFFFF;">1.</b> Segmentasi yang berguna berangkat dari <b style="color:#FFFFFF;">perilaku</b>, bukan demografi.<br>
<b style="color:#FFFFFF;">2.</b> RFM hanya butuh tiga kolom yang dimiliki hampir semua bisnis: siapa, kapan, berapa.<br>
<b style="color:#FFFFFF;">3.</b> Skor kuintil membuat tiga satuan berbeda menjadi setara dan bisa dibandingkan.<br>
<b style="color:#FFFFFF;">4.</b> Nama segmen adalah jembatan antara analis dan tim bisnis.<br>
<b style="color:#FFFFFF;">5.</b> Analisis berhenti bukan di chart, melainkan di <b style="color:#FFFFFF;">keputusan beserta angkanya</b>.
</div>
<div style="color:#8FD8C6;font-size:13.5px;margin-top:20px;line-height:1.7;">
<b>Langkah lanjutan yang bisa dicoba sendiri:</b> hitung ulang RFM per kuartal untuk melihat perpindahan segmen
(<i>migration analysis</i>) &middot; bandingkan hasil aturan RFM dengan K-Means pada R/F/M yang sudah di-<i>log-transform</i>
&middot; tambahkan dimensi ke-4 seperti kategori produk favorit atau margin.
</div></div>